# Retrieval Pipeline — End-to-End Evaluation

Wires together the full `src/retrieval/` stack — `DenseRetriever`, `BM25Retriever`,
`HybridRetriever` (RRF), and `Reranker` — and runs 3 test questions through each stage.

**Four stages per question:**
1. **Dense only** — MiniLM bi-encoder cosine similarity (top 5)
2. **BM25 only** — Okapi BM25 term overlap, in-memory index (top 5)
3. **Hybrid RRF** — Dense + BM25 fused via Reciprocal Rank Fusion — retrieve top 20, display top 5
4. **Reranked** — FlashRank cross-encoder re-scores hybrid top-20 → final top 5

**Collection:** `rag_recursive` — 12,389 points, 512-char chunks, 64-char overlap.

**Test set design:**
- **Q1** — proper-noun / terminology query (*RAGAS faithfulness*) — exact keyword matching matters
- **Q2** — technical mechanism query (*multi-head attention*) — semantics dominate; both methods should agree
- **Q3** — factual lookup with specific numbers (*GDP Q4 2025*) — hardest test for dense-only retrieval


In [ ]:
import sys
import textwrap
import logging

sys.path.insert(0, '..')
logging.basicConfig(level=logging.WARNING)  # suppress model-load INFO logs

from configs.settings import AppConfig
from src.embedding import Embedder
from src.vectorstore import VectorStore
from src.retrieval import DenseRetriever, BM25Retriever, HybridRetriever, Reranker

cfg        = AppConfig()
embedder   = Embedder()
store      = VectorStore()
COLLECTION = cfg.collection_recursive   # 'rag_recursive'

print('Initialising components...')
print('  BM25Retriever scrolls all ~12k chunks from Qdrant to build an in-memory index.')
dense_retriever = DenseRetriever(embedder, store)
bm25_retriever  = BM25Retriever(store, COLLECTION)
hybrid          = HybridRetriever(dense_retriever, bm25_retriever, cfg)
reranker        = Reranker(cfg)
print('All components ready.')


def show(results, label, query, n=5):
    sep = '─' * 74
    print()
    print(sep)
    print(f'  {label:<12}  {query[:62]}...')
    print(sep)
    for i, r in enumerate(results[:n], 1):
        src     = r.metadata.get('filename', '?')
        idx     = r.metadata.get('chunk_index', '?')
        size    = r.metadata.get('chunk_size', '?')
        preview = r.chunk_text.strip().replace('\n', ' ')[:320]
        preview = textwrap.fill(
            preview, width=72,
            initial_indent='    ', subsequent_indent='    ')
        print()
        print(f'  [{i}] score={r.score:.4f}  {src}  chunk={idx}  ~{size}ch  ({r.retrieval_method})')
        print(preview)
    print()


---
## Q1: What is the RAGAS framework and how does it measure faithfulness?

**Why this question:** *RAGAS* is a proper noun — an acronym that MiniLM won't have a strong
semantic embedding for. *Faithfulness* is a specific RAGAS metric term. Together they test
whether exact keyword matching is necessary when queries mix proper nouns with domain terminology.


In [ ]:
Q1 = 'What is the RAGAS framework and how does it measure faithfulness?'

# ── Dense ──────────────────────────────────────────────────────────────────
q1_dense = dense_retriever.retrieve(Q1, COLLECTION, limit=5)
show(q1_dense, 'DENSE', Q1)

# ── BM25 ───────────────────────────────────────────────────────────────────
q1_bm25 = bm25_retriever.retrieve(Q1, limit=5)
show(q1_bm25, 'BM25', Q1)

# ── Hybrid RRF (top-20 fetched, top-5 shown) ───────────────────────────────
q1_hybrid_20 = hybrid.retrieve_rrf(Q1, COLLECTION, limit=20)
show(q1_hybrid_20, 'HYBRID RRF', Q1)

# ── Reranker (cross-encoder over the 20 hybrid candidates) ─────────────────
q1_reranked = reranker.rerank(Q1, q1_hybrid_20, top_n=5)
show(q1_reranked, 'RERANKED', Q1)


### Q1 Observations — actual results

**Dense** — `ragas.pdf` dominates top-3 (scores 0.650 → 0.532), with `rag_llm.pdf` appearing at
rank 4–5 (0.524, 0.517). Dense *does* find the right document — *RAGAS* and *faithfulness* have
enough semantic weight in MiniLM's space to distinguish `ragas.pdf` from generic RAG papers. The
pre-run concern about drift was overstated for this corpus.

**BM25** — `ragas.pdf` at rank 1 (23.2), `rag_llm.pdf` at rank 2 (21.4), then `ragas.pdf` again
at 3–4. The term *faithfulness* isn't rare enough to fully exclude `rag_llm.pdf`, which mentions
RAGAS in passing. Both methods find the right document.

**Hybrid RRF** — ⚠️ Surprise: `rag_llm.pdf` floats to **rank 1**. Because BM25 ranked it 2nd and
dense ranked it 4th–5th, it accumulates just enough combined RRF weight to edge out individual
`ragas.pdf` chunks. This is an RRF failure mode: when both methods partially agree on a *wrong*
document, fusion can promote it above better results. **Hybrid is not always better than its parts.**

**Reranked** — Corrects hybrid's mistake cleanly. All top-5 are `ragas.pdf` with scores
0.996 → 0.941. The cross-encoder jointly reads query + chunk and immediately identifies that
`rag_llm.pdf` only mentions RAGAS in passing; `ragas.pdf` chunks contain the actual framework
definition. The reranker acts as a safety net for RRF false positives.

**Key takeaway:** Hybrid RRF can *demote* correct results when a borderline document ranks in the
top-3 of both methods. The reranker is not just a precision booster — it's a correctness fix.

---
## Q2: How does multi-head self-attention work in the transformer architecture?

**Why this question:** *Multi-head attention* and *transformer architecture* are verbatim phrases
from `attention_need.pdf`. Both dense and BM25 should agree strongly here — this is the control
question. When both methods agree, the interesting question shifts to the **reranker**: does it
prefer the mechanism chunk (Q/K/V formulation) over the introductory paragraph?


In [ ]:
Q2 = 'How does multi-head self-attention work in the transformer architecture?'

# ── Dense ──────────────────────────────────────────────────────────────────
q2_dense = dense_retriever.retrieve(Q2, COLLECTION, limit=5)
show(q2_dense, 'DENSE', Q2)

# ── BM25 ───────────────────────────────────────────────────────────────────
q2_bm25 = bm25_retriever.retrieve(Q2, limit=5)
show(q2_bm25, 'BM25', Q2)

# ── Hybrid RRF ─────────────────────────────────────────────────────────────
q2_hybrid_20 = hybrid.retrieve_rrf(Q2, COLLECTION, limit=20)
show(q2_hybrid_20, 'HYBRID RRF', Q2)

# ── Reranked ───────────────────────────────────────────────────────────────
q2_reranked = reranker.rerank(Q2, q2_hybrid_20, top_n=5)
show(q2_reranked, 'RERANKED', Q2)


### Q2 Observations — actual results

**Dense** — All 5 results from `attention_need.pdf` — exactly as expected. Chunk 14 ranks 1st
(0.797). Dense understands the semantic neighbourhood of self-attention deeply; no other paper in
the corpus competes.

**BM25** — Also all `attention_need.pdf`. Chunk 5 ranks 1st (24.7). *Multi-head*, *self-attention*,
*transformer*, *architecture* are all verbatim phrases in the paper. Dense and BM25 agree on the
document but **disagree on which chunk is best** — dense prefers chunk 14, BM25 prefers chunk 5.

**Hybrid RRF** — Also all `attention_need.pdf`, now ordered by combined rank. Chunk 35 rises to
rank 1 (0.032) — it was ranked 2nd by dense and 3rd by BM25, giving it the best aggregate position.
Hybrid doesn't rescue a different document here (both already agreed), but it does change intra-doc
ordering.

**Reranked** — Chunk 35 stays at rank 1 (0.997). The reranked top-1 contains: *"In this work we
employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64."*
This is the concrete implementation of multi-head attention — exactly what "how does it work?" asks
for. Dense's top-1 (chunk 14) is likely a higher-level description; the reranker prefers the chunk
with the actual computation. Chunk 19 (not in hybrid's top-4) appears at reranked rank 3 — the
cross-encoder pulled it up from deeper in the 20-candidate set.

**Key takeaway:** Even when dense and BM25 agree on the document, the reranker changes *which
chunk* wins. Dense found a good chunk; the reranker found the *most directly answering* chunk.

---
## Q3: What was the GDP growth rate for the United States in Q4 2025?

**Why this question:** This is a factual lookup with domain-specific numbers. *GDP*, *Q4*, *2025*,
and *United States* are exact, high-discrimination terms that only appear in `gdp4q25.pdf`. Dense
similarity over an ML paper corpus will likely drift — *growth rate* sits semantically close to
*model performance metrics*, *F1 improvement*, etc. This is the canonical demonstration of why
hybrid > dense-alone for factual queries.


In [ ]:
Q3 = 'What was the GDP growth rate for the United States in Q4 2025?'

# ── Dense ──────────────────────────────────────────────────────────────────
q3_dense = dense_retriever.retrieve(Q3, COLLECTION, limit=5)
show(q3_dense, 'DENSE', Q3)

# ── BM25 ───────────────────────────────────────────────────────────────────
q3_bm25 = bm25_retriever.retrieve(Q3, limit=5)
show(q3_bm25, 'BM25', Q3)

# ── Hybrid RRF ─────────────────────────────────────────────────────────────
q3_hybrid_20 = hybrid.retrieve_rrf(Q3, COLLECTION, limit=20)
show(q3_hybrid_20, 'HYBRID RRF', Q3)

# ── Reranked ───────────────────────────────────────────────────────────────
q3_reranked = reranker.rerank(Q3, q3_hybrid_20, top_n=5)
show(q3_reranked, 'RERANKED', Q3)


### Q3 Observations — actual results

**Dense** — All 5 results from `gdp4q25.pdf` (scores 0.676 → 0.632). The pre-run prediction of
semantic drift was wrong for this corpus. The economics document is sufficiently distant from the
ML papers in embedding space that MiniLM clearly routes it to `gdp4q25.pdf`. Corpus size matters:
with only 14 documents, there's no ML paper that's semantically close enough to compete.

**BM25** — Also all `gdp4q25.pdf` (scores 25.7 → 25.4). *GDP* is unique to that document; *Q4*,
*2025*, and *United States* are highly discriminative. Both methods agree completely.

**Hybrid RRF** — All `gdp4q25.pdf`. When both methods agree perfectly, fusion produces no change
to document composition — the result is identical to what either method would return alone.

**Reranked** — All `gdp4q25.pdf` with near-perfect scores (0.999, 0.998, 0.996). The cross-encoder
is highly confident — the chunks directly contain the GDP figure with the exact terminology from
the query.

**Key takeaway:** For a small corpus with domain-distinct documents, dense retrieval alone can match
BM25 for factual lookups. The semantic drift failure mode is a *scale problem* — it becomes
pronounced in large corpora where many documents discuss performance, growth, and improvement in
different domains. This corpus is too small to stress-test that failure. **Rerun Q3 on a larger
corpus (e.g. add financial news articles alongside ML papers) to see semantic drift in practice.**

---
## Pipeline Summary — Findings

### What each stage actually contributed across the 3 questions

| Stage | Q1 (RAGAS) | Q2 (Attention) | Q3 (GDP) |
|-------|------------|----------------|----------|
| **Dense** | ✅ Found right doc | ✅ Found right doc (wrong chunk rank) | ✅ Found right doc |
| **BM25** | ✅ Found right doc | ✅ Found right doc (wrong chunk rank) | ✅ Found right doc |
| **Hybrid RRF** | ❌ Promoted wrong doc to rank 1 | ✅ Improved intra-doc ordering | ✅ No change needed |
| **Reranker** | ✅ Corrected hybrid's mistake | ✅ Promoted best-answering chunk | ✅ Confirmed top result |

### Three patterns observed

**1. RRF false-positive (Q1)** — Hybrid can promote a document that both methods *partially* agree
on, even if it's not the best answer. When a borderline document ranks 2nd in BM25 and 4th in
dense, its combined RRF score beats the actual answer. The reranker is the fix.

**2. Intra-document reranking (Q2)** — When both methods agree on the *document*, they still
disagree on the *chunk*. Dense ranked chunk 14 first (introductory), BM25 ranked chunk 5 first
(terminology-dense intro). The reranker promoted chunk 35 — the one with the actual h=8 computation.
This is the core value of cross-encoding: it reads the question and the passage together.

**3. Corpus scale caveat (Q3)** — Semantic drift is a scale problem. With 14 documents that span
clearly distinct domains (ML papers vs economics report), even a bi-encoder routes correctly. In
a production corpus with thousands of documents across overlapping domains, Q3's failure mode
would appear. The hybrid pipeline is built for that scale; this corpus is too small to stress-test it.

### Implications for the evaluation dashboard

Before computing RAGAS metrics, the Q1 finding sets a clear expectation:
- **Faithfulness** and **context precision** should be measured on *reranked* results, not raw hybrid
- Raw hybrid rank-1 can be wrong even when both retrieval methods are individually correct
- The pipeline is: `hybrid(top-20)` → `reranker(top-5)` → LLM — never skip the reranker step

**MRR lift from reranking** is the metric to watch: compare the rank of the correct chunk in
hybrid top-20 vs its rank after reranking. Q1 shows MRR going from 0 (wrong doc at rank 1) to 1.0
(correct doc at rank 1). That's the strongest possible lift.